![image_1781199219040.png](./image_1781199219040.png "image_1781199219040.png")

![image_1781199252113.png](./image_1781199252113.png "image_1781199252113.png")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f

# Initialize Spark session
spark = SparkSession.builder.appName("EventsRegistrations").getOrCreate()

# =========================
# Events Table
# =========================
events_data = [
    (1, "Tech Conference", 200),
    (2, "Music Festival", 500),
    (3, "Art Workshop", 30),
]

events_columns = ["event_id", "event_name", "capacity"]

events_df = spark.createDataFrame(events_data, events_columns)

# =========================
# Registrations Table
# =========================
registrations_data = [
    (1, 1, 101, 1),
    (2, 1, 102, 1),
    (3, 1, 103, 1),
    (4, 1, 104, 1),
    (5, 1, 105, 0),
    (6, 2, 201, 1),
    (7, 2, 202, 1),
    (8, 2, 203, 0),
    (9, 2, 204, 1),
    (10, 2, 205, 0),
    (11, 2, 206, 0),
    (12, 2, 207, 1),
    (13, 3, 301, 1),
    (14, 3, 302, 0),
    (15, 3, 303, 0),
    (16, 3, 304, None),  # attended missing, so we use None
]

registrations_columns = ["reg_id", "event_id", "user_id", "attended"]

registrations_df = spark.createDataFrame(registrations_data, registrations_columns)

# =========================
# Show DataFrames
# =========================
print("Events Table:")
events_df.show()

print("Registrations Table:")
registrations_df.show()

In [0]:
result_df = (
    events_df.join(registrations_df, on="event_id", how="inner")
    .groupBy("event_name")
    .agg(
        f.count("*").alias("total_registrations"),
        f.sum("attended").alias("total_attended"),
    )
    .select(
        f.col("event_name"),
        f.round(f.col("total_attended") * 100 / f.col("total_registrations"), 1).alias(
            "attendance_rate"
        ),
    )
    .orderBy(f.desc("attendance_rate"),f.asc("event_name"))
)

display(result_df)